In [ ]:

!apt-get update -qq
!apt-get install -y rabbitmq-server


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
rabbitmq-server is already the newest version (3.12.1-1ubuntu1.7).
0 upgraded, 0 newly installed, 0 to remove and 40 not upgraded.


In [ ]:

!service rabbitmq-server start


[ OK ]


In [ ]:

!pip install pika


In [ ]:

import os

os.makedirs("stream", exist_ok=True)

print("Stream folder created!")


Stream folder created!


In [ ]:

%%writefile stream/dispatcher.py

import pika
import json
import time
import random
import uuid
from datetime import datetime

# 1. Connect to RabbitMQ
connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)

channel = connection.channel()

# 2. Create topic exchange
channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

# 3. Define cities and their zones
zones = {
    "Bangalore": ["koramangala", "indiranagar", "whitefield"],
    "Chennai": ["tnagar", "adyar", "velachery"]
}

# 4. Define request types
types = ["ride", "food"]


# 5. Generate messages continuously
message_count = 0

try:
    while True:
        message_count += 1

        # Randomly select city
        city = random.choice(list(zones.keys()))

        # Select a zone belonging to that city
        zone = random.choice(zones[city])

        # Select ride or food
        request_type = random.choice(types)

        # Build routing key BEFORE modifying the message
        routing_key = (
            city.lower() + "." + zone + "." + request_type
        )

        # Create JSON message
        message = {
            "id": str(uuid.uuid4()),
            "city": city,
            "zone": zone,
            "type": request_type,
            "ts": datetime.now().isoformat()
        }

        # Every 10th message is BAD (missing zone)
        if message_count % 10 == 0:
            message.pop("zone")

            print(
                f"BAD MESSAGE #{message_count}: zone removed",
                flush=True
            )
        else:
            print(
                f"GOOD MESSAGE #{message_count}",
                flush=True
            )

        # Publish message to exchange
        channel.basic_publish(
            exchange="rapido-dispatch",
            routing_key=routing_key,
            body=json.dumps(message)
        )

        print("Published:", routing_key, message, flush=True)

        # Wait for 2 seconds
        time.sleep(2)

except KeyboardInterrupt:
    print("Dispatcher stopped")

finally:
    connection.close()


Overwriting stream/dispatcher.py


In [ ]:

%%writefile stream/driver.py

import pika
import sys
import json

# 1. Get driver name and binding pattern
name = sys.argv[1]
pattern = sys.argv[2]

# 2. Connect to RabbitMQ
connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)

channel = connection.channel()

# 3. Declare topic exchange
channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

# 4. Create a private queue
result = channel.queue_declare(
    queue="",
    exclusive=True
)

queue_name = result.method.queue

# 5. Bind private queue to exchange
channel.queue_bind(
    exchange="rapido-dispatch",
    queue=queue_name,
    routing_key=pattern
)

print(
    f"{name} listening with pattern: {pattern}",
    flush=True
)

def callback(ch, method, properties, body):
    message = json.loads(body)

    if "zone" not in message:
        print(
            f"[{name}] BAD MESSAGE - NACK "
            f"(id={message.get('id')})",
            flush=True
        )

        ch.basic_nack(
            delivery_tag=method.delivery_tag,
            requeue=False
        )

    else:
        print(
            f"[{name}] GOOD MESSAGE - ACK "
            f"(id={message.get('id')}, zone={message['zone']})",
            flush=True
        )

        ch.basic_ack(
            delivery_tag=method.delivery_tag
        )
# 6. Start consuming messages
channel.basic_consume(
    queue=queue_name,
    on_message_callback=callback,
    auto_ack=False
)

try:
    channel.start_consuming()

except KeyboardInterrupt:
    print(f"{name} stopped")

finally:
    connection.close()

Overwriting stream/driver.py


In [ ]:

import subprocess
import sys
import time

# Store output files
driver_logs = {}

# Driver 1: Ravi
driver_logs["Ravi"] = open("ravi.txt", "w")

ravi = subprocess.Popen(
    [
        sys.executable, "-u",
        "stream/driver.py",
        "Ravi",
        "bangalore.koramangala.*"
    ],
    stdout=driver_logs["Ravi"],
    stderr=subprocess.STDOUT
)

# Driver 2: Meena
driver_logs["Meena"] = open("meena.txt", "w")

meena = subprocess.Popen(
    [
        sys.executable, "-u",
        "stream/driver.py",
        "Meena",
        "bangalore.*.food"
    ],
    stdout=driver_logs["Meena"],
    stderr=subprocess.STDOUT
)

# Driver 3: Control
driver_logs["Control"] = open("control.txt", "w")

control = subprocess.Popen(
    [
        sys.executable, "-u",
        "stream/driver.py",
        "Control",
        "bangalore.#"
    ],
    stdout=driver_logs["Control"],
    stderr=subprocess.STDOUT
)

# Give drivers time to connect and bind queues
time.sleep(3)

print("All three drivers have started!")


All three drivers have started!


In [ ]:

import subprocess
import sys

print("Dispatcher running for 2 minutes...")

with open("dispatcher_output.txt", "w") as output:

    result = subprocess.run(
        [
            "timeout",
            "120s",
            sys.executable,
            "-u",
            "stream/dispatcher.py"
        ],
        stdout=output,
        stderr=subprocess.STDOUT
    )

print("Two-minute run completed!")


Dispatcher running for 2 minutes...
Two-minute run completed!


In [ ]:

with open("dispatcher_output.txt", "r") as f:
    print(f.read()[:3000])


GOOD MESSAGE #1
Published: bangalore.koramangala.food {'id': '9dc9c138-affd-4b44-9d6d-7bb2f5098c75', 'city': 'Bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-10T18:17:23.978637'}
GOOD MESSAGE #2
Published: bangalore.whitefield.food {'id': 'c7a38b35-c68e-4226-8ebd-895085a25545', 'city': 'Bangalore', 'zone': 'whitefield', 'type': 'food', 'ts': '2026-11-10T18:17:25.979327'}
GOOD MESSAGE #3
Published: chennai.velachery.food {'id': '4803ceb1-2a0e-47bb-8ae8-0ae287a5173f', 'city': 'Chennai', 'zone': 'velachery', 'type': 'food', 'ts': '2026-11-10T18:17:27.980406'}
GOOD MESSAGE #4
Published: bangalore.indiranagar.food {'id': 'ba67110b-5a82-449d-a148-ff185579a2e8', 'city': 'Bangalore', 'zone': 'indiranagar', 'type': 'food', 'ts': '2026-11-10T18:17:29.981596'}
GOOD MESSAGE #5
Published: bangalore.whitefield.food {'id': 'd84588f4-ad56-4a99-bfef-43e0950caa29', 'city': 'Bangalore', 'zone': 'whitefield', 'type': 'food', 'ts': '2026-11-10T18:17:31.982530'}
GOOD MESSAGE #6
Published: 

In [ ]:

for driver in ["Ravi", "Meena", "Control"]:

    filename = driver.lower() + ".txt"

    print("\n==========", driver, "==========")

    with open(filename, "r") as f:
        print(f.read()[:2000])



========== Ravi ==========
Ravi listening with pattern: bangalore.koramangala.*
[Ravi] GOOD MESSAGE - ACK (id=9dc9c138-affd-4b44-9d6d-7bb2f5098c75, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=36812971-33d3-4051-adcf-1cb554222b53, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=cf1bf0ad-554f-4745-b751-266b7ae485b9, zone=koramangala)
[Ravi] BAD MESSAGE - NACK (id=e800c039-aa84-458d-ae15-487129e7f76c)
[Ravi] GOOD MESSAGE - ACK (id=56ea9e0a-6eae-486b-80c9-c69f556c9f69, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=a2acf1e6-61a0-4cf9-a5d5-77d787635491, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=ecc8d3d4-7b5f-4f5a-be7d-94038b6066a7, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=5c6a7c80-de90-412b-b42a-0f2b41e160ca, zone=koramangala)
Ravi stopped


========== Meena ==========
Meena listening with pattern: bangalore.*.food
[Meena] GOOD MESSAGE - ACK (id=9dc9c138-affd-4b44-9d6d-7bb2f5098c75, zone=koramangala)
[Meena] GOOD MESSAGE - ACK (id=c7a38b35-c68e-4226-8ebd-895085a25545, zon

In [ ]:

import pika
import json
import time
from datetime import datetime

# Connect to RabbitMQ
connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)

channel = connection.channel()

# Create message
message = {
    "id": "TEST-001",
    "city": "Bangalore",
    "zone": "koramangala",
    "type": "food",
    "ts": datetime.now().isoformat()
}

# Publish test message
channel.basic_publish(
    exchange="rapido-dispatch",
    routing_key="bangalore.koramangala.food",
    body=json.dumps(message)
)

print("Test message published successfully!")

connection.close()

# Wait for drivers to receive it
time.sleep(2)


Test message published successfully!


In [ ]:

for driver in ["Ravi", "Meena", "Control"]:

    filename = driver.lower() + ".txt"

    with open(filename, "r") as f:
        content = f.read()

    print("\n", driver)

    if "TEST-001" in content:
        print("Received TEST-001")
    else:
        print("Did not receive TEST-001")



 Ravi
Did not receive TEST-001

 Meena
Did not receive TEST-001

 Control
Did not receive TEST-001


In [ ]:

for process in [ravi, meena, control]:

    process.terminate()

print("All drivers stopped.")

for f in driver_logs.values():
    f.close()


All drivers stopped.


In [ ]:

for driver in ["Ravi", "Meena", "Control"]:

    filename = driver.lower() + ".txt"

    print("\n==========", driver, "==========")

    with open(filename, "r") as f:
        for line in f:
            if "TEST-001" in line:
                print(line.strip())



========== Ravi ==========

========== Meena ==========

========== Control ==========


In [ ]:
with open("dispatcher_output.txt", "r") as f:
    print(f.read()[-3000:])


5
Published: bangalore.whitefield.food {'id': '79d9a0d2-43b8-4c46-a3a2-dd1a0253d731', 'city': 'Bangalore', 'zone': 'whitefield', 'type': 'food', 'ts': '2026-11-10T18:00:37.776320'}
GOOD MESSAGE #46
Published: chennai.velachery.food {'id': '0a75881e-470b-4941-b524-5c46d9042dd2', 'city': 'Chennai', 'zone': 'velachery', 'type': 'food', 'ts': '2026-11-10T18:00:39.777426'}
GOOD MESSAGE #47
Published: chennai.tnagar.ride {'id': '4c674127-0637-42eb-8077-4e42a6be871a', 'city': 'Chennai', 'zone': 'tnagar', 'type': 'ride', 'ts': '2026-11-10T18:00:41.778502'}
GOOD MESSAGE #48
Published: chennai.tnagar.ride {'id': '66ccaf94-d653-4969-a92f-1c14a2322321', 'city': 'Chennai', 'zone': 'tnagar', 'type': 'ride', 'ts': '2026-11-10T18:00:43.779358'}
GOOD MESSAGE #49
Published: bangalore.koramangala.ride {'id': '367b6b7e-578c-4c4d-bb7b-3bfe291b3af5', 'city': 'Bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-10T18:00:45.780261'}
BAD MESSAGE #50: zone removed
Published: bangalore.whitefield.f

In [ ]:
!ps aux | grep driver.py

root       62966  0.0  0.0   7344  3632 ?        S    18:03   0:00 /bin/bash -c ps aux | grep driver.py
root       62968  0.0  0.0   6680  2360 ?        S    18:03   0:00 grep driver.py


In [ ]:
!ps aux | grep '[d]river.py'

In [ ]:
for name in ["Ravi", "Meena", "Control"]:
    print("\n==========", name, "==========")
    with open(name.lower() + ".txt", "r") as f:
        print(f.read())


========== Ravi ==========
Ravi listening with pattern: bangalore.koramangala.*


========== Meena ==========
Meena listening with pattern: bangalore.*.food


========== Control ==========
Control listening with pattern: bangalore.#



In [ ]:
!pgrep -af "stream/driver.py"

In [ ]:
for p in processes:
    print("Process ID:", p.pid, "| Status:", p.poll())

Process ID: 65301 | Status: 0
Process ID: 65302 | Status: 0
Process ID: 65305 | Status: 0


In [ ]:
!tail -20 stream/driver.py

        )

        ch.basic_ack(
            delivery_tag=method.delivery_tag
        )
# 6. Start consuming messages
channel.basic_consume(
    queue=queue_name,
    on_message_callback=callback,
    auto_ack=False
)

try:
    channel.start_consuming()

except KeyboardInterrupt:
    print(f"{name} stopped")

finally:
    connection.close()


In [ ]:
for name in ["Ravi", "Meena", "Control"]:
    print("\n==========", name, "==========")
    print(open(name.lower() + ".txt").read())


========== Ravi ==========
Ravi listening with pattern: bangalore.koramangala.*
[Ravi] GOOD MESSAGE - ACK (id=9dc9c138-affd-4b44-9d6d-7bb2f5098c75, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=36812971-33d3-4051-adcf-1cb554222b53, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=cf1bf0ad-554f-4745-b751-266b7ae485b9, zone=koramangala)
[Ravi] BAD MESSAGE - NACK (id=e800c039-aa84-458d-ae15-487129e7f76c)
[Ravi] GOOD MESSAGE - ACK (id=56ea9e0a-6eae-486b-80c9-c69f556c9f69, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=a2acf1e6-61a0-4cf9-a5d5-77d787635491, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=ecc8d3d4-7b5f-4f5a-be7d-94038b6066a7, zone=koramangala)
[Ravi] GOOD MESSAGE - ACK (id=5c6a7c80-de90-412b-b42a-0f2b41e160ca, zone=koramangala)
Ravi stopped


========== Meena ==========
Meena listening with pattern: bangalore.*.food
[Meena] GOOD MESSAGE - ACK (id=9dc9c138-affd-4b44-9d6d-7bb2f5098c75, zone=koramangala)
[Meena] GOOD MESSAGE - ACK (id=c7a38b35-c68e-4226-8ebd-895085a25545, zon